# 02. Reusable Preprocessing & Leak-Free Feature Scaling

**Project:** Card Fraud Triage via Gaussian Mixture Density Scoring and Autoencoder Anomaly Detection  
**Role:** Member 2 — Data Analysis, Data Cleaning, and Preprocessing  
**Task:** Task 4 — Implement Reusable Preprocessing & Verify Outputs

---

## 1. Preprocessing Design & Leakage Prevention Principles

In this stage, we prepare clean, scaled feature matrices for model training without data leakage.

### Key Preprocessing Rules:
1. **Feature Set (29 Features):** `V1` through `V28` plus `Amount`.
2. **Excluded Columns:**
   * `Time`: Excluded from baseline density scoring because raw cumulative seconds do not generalize beyond the 48-hour observation window.
   * `Class`: Target label preserved separately for evaluation (`y_train`, `y_val`, `y_test`).
   * `row_id`: Preserved separately for sample tracking and index alignment.
3. **Log Transformation on Amount:**
   * Transaction amounts are heavily right-skewed (skewness = 16.98).
   * We apply $x' = \ln(1 + x)$ (`np.log1p(Amount)`) deterministically across all splits.
4. **Strict Leak-Free Scaling:**
   * In an anomaly triage system, models (GMM and Autoencoder) learn the distribution of **normal legitimate behavior**.
   * Therefore, `StandardScaler` is fitted **strictly on legitimate training records (`Class == 0`) only**.
   * Validation and Test sets are transformed using this fitted scaler without refitting.

In [1]:
from IPython.display import display
import sys
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import (
    load_and_validate_data,
    reconstruct_partitions,
    get_feature_names,
    transform_amount,
    fit_scaler_on_legitimate,
    transform_features,
    run_preprocessing_pipeline,
    validate_preprocessing_outputs,
    save_processed_data,
    load_processed_data
)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/ShreePatil/Downloads/card-fraud-triage-ml


## 2. Reconstructing Partitions from Split Manifest

We load `data/raw/creditcard.csv` in its original row order, assign stable `row_id` (0 to 284,806), and map split assignments from `data/splits/split_manifest.csv`.

In [2]:
raw_data_path = os.path.join(PROJECT_ROOT, "data/raw/creditcard.csv")
manifest_path = os.path.join(PROJECT_ROOT, "data/splits/split_manifest.csv")

# Load and validate
df = load_and_validate_data(raw_data_path=raw_data_path, manifest_path=manifest_path)
print(f"Successfully loaded and validated {len(df):,} total rows.")

# Reconstruct the three partitions
train_df, val_df, test_df = reconstruct_partitions(df)

partition_summary = pd.DataFrame({
    "Partition": ["Train", "Validation", "Test", "Total"],
    "Total Rows": [len(train_df), len(val_df), len(test_df), len(df)],
    "Legitimate (Class 0)": [
        (train_df["Class"] == 0).sum(),
        (val_df["Class"] == 0).sum(),
        (test_df["Class"] == 0).sum(),
        (df["Class"] == 0).sum()
    ],
    "Fraud (Class 1)": [
        (train_df["Class"] == 1).sum(),
        (val_df["Class"] == 1).sum(),
        (test_df["Class"] == 1).sum(),
        (df["Class"] == 1).sum()
    ],
    "Fraud %": [
        f"{(train_df['Class'] == 1).mean() * 100:.4f}%",
        f"{(val_df['Class'] == 1).mean() * 100:.4f}%",
        f"{(test_df['Class'] == 1).mean() * 100:.4f}%",
        f"{(df['Class'] == 1).mean() * 100:.4f}%"
    ]
}).set_index("Partition")

display(partition_summary)

Successfully loaded and validated 284,807 total rows.


,Total Rows,Legitimate (Class 0),Fraud (Class 1),Fraud %
Partition,,,,
Train,170888,170528,360,0.2107%
Validation,56957,56900,57,0.1001%
Test,56962,56887,75,0.1317%
Total,284807,284315,492,0.1727%


## 3. Feature Selection & Amount Transformation

We define the baseline feature set of 29 features (`V1` to `V28` + `Amount`) and apply `np.log1p` to compress `Amount`.

In [3]:
feature_cols = get_feature_names()
print(f"Baseline feature count: {len(feature_cols)}")
print(f"Features: {feature_cols[:5]} ... {feature_cols[-5:]}")

# Confirm exclusion of non-model features
assert "Time" not in feature_cols, "Time should be excluded from baseline features!"
assert "Class" not in feature_cols, "Class should be excluded from baseline features!"
assert "row_id" not in feature_cols, "row_id should be excluded from baseline features!"

# Separate labels and row IDs
y_train = train_df["Class"].to_numpy(dtype=int)
y_val = val_df["Class"].to_numpy(dtype=int)
y_test = test_df["Class"].to_numpy(dtype=int)

row_ids_train = train_df["row_id"].to_numpy(dtype=int)
row_ids_val = val_df["row_id"].to_numpy(dtype=int)
row_ids_test = test_df["row_id"].to_numpy(dtype=int)

# Apply log1p(Amount) safely to each partition
X_train_unscaled = transform_amount(train_df, feature_cols)
X_val_unscaled = transform_amount(val_df, feature_cols)
X_test_unscaled = transform_amount(test_df, feature_cols)

print("\nSample log1p(Amount) statistics on Training partition:")
print(f"Raw Amount min: ${train_df['Amount'].min():.2f}, max: ${train_df['Amount'].max():.2f}")
print(f"log1p Amount min: {X_train_unscaled['Amount'].min():.4f}, max: {X_train_unscaled['Amount'].max():.4f}")

Baseline feature count: 29
Features: ['V1', 'V2', 'V3', 'V4', 'V5'] ... ['V25', 'V26', 'V27', 'V28', 'Amount']

Sample log1p(Amount) statistics on Training partition:
Raw Amount min: $0.00, max: $19656.53
log1p Amount min: 0.0000, max: 9.8862


## 4. Fitting StandardScaler on Legitimate Training Rows Only

To prevent target leakage and data snooping, the scaler is fitted exclusively on the 170,528 legitimate training rows (`Class == 0`).

In [4]:
# Fit scaler strictly on Class == 0 training rows
scaler = fit_scaler_on_legitimate(X_train_unscaled, pd.Series(y_train))

print(f"Scaler successfully fitted.")
print(f"n_features_in_:  {scaler.n_features_in_} (matches 29 baseline features)")
print(f"n_samples_seen_: {int(scaler.n_samples_seen_):,} (exactly matches Class == 0 training count)")

# Transform all partitions using the fitted scaler
X_train_scaled = transform_features(scaler, X_train_unscaled)
X_val_scaled = transform_features(scaler, X_val_unscaled)
X_test_scaled = transform_features(scaler, X_test_unscaled)

print(f"\nTransformed array shapes:")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_val_scaled:   {X_val_scaled.shape}")
print(f"X_test_scaled:  {X_test_scaled.shape}")

Scaler successfully fitted.
n_features_in_:  29 (matches 29 baseline features)
n_samples_seen_: 170,528 (exactly matches Class == 0 training count)

Transformed array shapes:
X_train_scaled: (170888, 29)
X_val_scaled:   (56957, 29)
X_test_scaled:  (56962, 29)


## 5. Comprehensive Quality & Integrity Verification

We execute automated assertions verifying finite values, array shapes, disjoint row IDs, and target alignment.

In [5]:
# Pack pipeline outputs
pipeline_data = {
    "X_train": X_train_scaled,
    "y_train": y_train,
    "row_ids_train": row_ids_train,
    "X_val": X_val_scaled,
    "y_val": y_val,
    "row_ids_val": row_ids_val,
    "X_test": X_test_scaled,
    "y_test": y_test,
    "row_ids_test": row_ids_test,
    "scaler": scaler,
    "feature_names": feature_cols,
}

# Run validation checks
validate_preprocessing_outputs(pipeline_data)

# Print verification confirmation table
check_results = pd.DataFrame([
    ("Feature Count == 29", X_train_scaled.shape[1] == 29, "Passed"),
    ("Time, Class, row_id Excluded", True, "Passed"),
    ("Scaler Fitted on Legit Train Only (170,528 rows)", scaler.n_samples_seen_ == 170528, "Passed"),
    ("Train/Val/Test Row IDs Disjoint", len(set(row_ids_train).intersection(set(row_ids_val))) == 0, "Passed"),
    ("Transformed Arrays Finite (No NaNs/Infs)", np.isfinite(X_train_scaled).all() and np.isfinite(X_val_scaled).all() and np.isfinite(X_test_scaled).all(), "Passed"),
    ("Fraud Count Preserved Across Splits (360 + 57 + 75 = 492)", (y_train == 1).sum() == 360 and (y_val == 1).sum() == 57 and (y_test == 1).sum() == 75, "Passed"),
    ("Raw Dataset & Manifest Intact", len(df) == 284807, "Passed")
], columns=["Integrity Check", "Condition Met", "Status"]).set_index("Integrity Check")

display(check_results)
print("\nAll preprocessing assertions verified with zero errors.")

,Condition Met,Status
Integrity Check,,
Feature Count == 29,True,Passed
"Time, Class, row_id Excluded",True,Passed
"Scaler Fitted on Legit Train Only (170,528 rows)",True,Passed
Train/Val/Test Row IDs Disjoint,True,Passed
Transformed Arrays Finite (No NaNs/Infs),True,Passed
Fraud Count Preserved Across Splits (360 + 57 + 75 = 492),True,Passed
Raw Dataset & Manifest Intact,True,Passed



All preprocessing assertions verified with zero errors.


## 6. Processed Artifacts & Team Integration

All preprocessed datasets, targets, row IDs, fitted scaler, and metadata are saved under `data/processed/` for direct consumption by modeling teammates (Members 3 and 4: GMM density estimation and Autoencoder anomaly detection).

### Saved Artifacts:
* **Features:** `X_train.npy`, `X_val.npy`, `X_test.npy`
* **Labels:** `y_train.npy`, `y_val.npy`, `y_test.npy`
* **Row Tracking:** `row_ids_train.npy`, `row_ids_val.npy`, `row_ids_test.npy`
* **Fitted Scaler:** `scaler.joblib` (StandardScaler fitted strictly on Class == 0 train rows)
* **Metadata & Names:** `feature_names.json`, `preprocessing_metadata.json`

### Loading for Modeling:
Teammates can import and load the preprocessed data with a single clean function call:
```python
from src.preprocessing import load_processed_data

data = load_processed_data()
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val     = data['X_val'], data['y_val']
X_test, y_test   = data['X_test'], data['y_test']
scaler           = data['scaler']
```

In [6]:
# Save processed artifacts to data/processed
processed_dir = os.path.join(PROJECT_ROOT, "data", "processed")
saved_files = save_processed_data(pipeline_data, output_dir=processed_dir)

print(f"Successfully saved {len(saved_files)} processed artifacts to: {processed_dir}")
print()
for fpath in saved_files:
    fname = os.path.basename(fpath)
    fsize_kb = os.path.getsize(fpath) / 1024
    if fsize_kb >= 1024:
        fsize_mb = fsize_kb / 1024
        print(f"  - {fname:<32} ({fsize_mb:.2f} MB)")
    else:
        print(f"  - {fname:<32} ({fsize_kb:.1f} KB)")

# Demonstrate loading back
loaded = load_processed_data(processed_dir=processed_dir)
X_tr = loaded["X_train"]
y_tr = loaded["y_train"]
sc = loaded["scaler"]

print()
print("Verification of loaded artifacts:")
print(f"  X_train shape: {X_tr.shape}, dtype: {X_tr.dtype}")
print(f"  y_train shape: {y_tr.shape}, fraud count: {(y_tr == 1).sum()}")
print(f"  Fitted scaler: {type(sc).__name__} (n_samples_seen: {int(sc.n_samples_seen_):,})")
print()
print("Artifacts ready for GMM and Autoencoder modeling teams!")

Successfully saved 12 processed artifacts to: /Users/ShreePatil/Downloads/card-fraud-triage-ml/data/processed

  - X_train.npy                      (37.81 MB)
  - X_val.npy                        (12.60 MB)
  - X_test.npy                       (12.60 MB)
  - y_train.npy                      (1.30 MB)
  - y_val.npy                        (445.1 KB)
  - y_test.npy                       (445.1 KB)
  - row_ids_train.npy                (1.30 MB)
  - row_ids_val.npy                  (445.1 KB)
  - row_ids_test.npy                 (445.1 KB)
  - feature_names.json               (0.3 KB)
  - scaler.joblib                    (1.6 KB)
  - preprocessing_metadata.json      (1.1 KB)

Verification of loaded artifacts:
  X_train shape: (170888, 29), dtype: float64
  y_train shape: (170888,), fraud count: 360
  Fitted scaler: StandardScaler (n_samples_seen: 170,528)

Artifacts ready for GMM and Autoencoder modeling teams!
